In [1]:

from pathlib import Path
import subprocess
import shutil
import time

BUCKET = "dcceew-eds-data"
S3_ROOT = "EDS-work-testing/eds-work-processing"
LOCAL_ROOT = Path("/home/jovyan/scratch/eds-work-processing")

def run_cmd(cmd, allow_empty_ls=False):
    print(" ".join(map(str, cmd)))

    result = subprocess.run(
        cmd,
        text=True,
        capture_output=True,
    )

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print(result.stderr)

    output = (result.stdout or "") + (result.stderr or "")

    if result.returncode != 0:
        if allow_empty_ls and "Total Objects: 0" in output:
            print("[OK] Prefix exists or is empty. No files found yet.")
            return result

        raise RuntimeError(
            f"Command failed: {result.returncode}\n\n"
            f"Command was:\n{' '.join(map(str, cmd))}\n\n"
            f"Output was:\n{output}"
        )

    return result


def s3_uri(tile=None, run=None):
    parts = [S3_ROOT.strip("/")]

    if tile:
        parts.append(tile.strip("/"))

    if run:
        parts.append(run.strip("/"))

    return f"s3://{BUCKET}/" + "/".join(parts) + "/"


def local_path(tile=None, run=None):
    p = LOCAL_ROOT

    if tile:
        p = p / tile

    if run:
        p = p / run

    return p


def check_local(tile=None, run=None):
    p = local_path(tile, run)

    print(f"Checking local path: {p}")

    if not p.exists():
        raise FileNotFoundError(f"Local path does not exist: {p}")

    if not p.is_dir():
        raise NotADirectoryError(f"Local path is not a directory: {p}")

    files = [x for x in p.rglob("*") if x.is_file()]
    total_bytes = sum(x.stat().st_size for x in files)

    print(f"Files: {len(files)}")
    print(f"Size: {total_bytes / 1e9:.2f} GB")

    print("First few files:")
    for f in files[:10]:
        print(" -", f)

    return p, len(files), total_bytes


def list_s3(tile=None, run=None):
    return run_cmd(
        [
            "aws", "s3", "ls",
            s3_uri(tile, run),
            "--recursive",
            "--human-readable",
            "--summarize",
        ],
        allow_empty_ls=True,
    )


def push_tile(tile, dryrun=True):
    check_local(tile)

    cmd = [
        "aws", "s3", "sync",
        str(local_path(tile)) + "/",
        s3_uri(tile),
        "--only-show-errors",
        "--no-progress",
    ]

    if dryrun:
        cmd.append("--dryrun")

    return run_cmd(cmd)


def push_run(tile, run, dryrun=True):
    check_local(tile, run)

    cmd = [
        "aws", "s3", "sync",
        str(local_path(tile, run)) + "/",
        s3_uri(tile, run),
        "--only-show-errors",
        "--no-progress",
    ]

    if dryrun:
        cmd.append("--dryrun")

    return run_cmd(cmd)


def pull_tile(tile, dryrun=True):
    cmd = [
        "aws", "s3", "sync",
        s3_uri(tile),
        str(local_path(tile)) + "/",
        "--only-show-errors",
        "--no-progress",
    ]

    if dryrun:
        cmd.append("--dryrun")

    return run_cmd(cmd)


def pull_run(tile, run, dryrun=True):
    cmd = [
        "aws", "s3", "sync",
        s3_uri(tile, run),
        str(local_path(tile, run)) + "/",
        "--only-show-errors",
        "--no-progress",
    ]

    if dryrun:
        cmd.append("--dryrun")

    return run_cmd(cmd)


print("Upload/download helpers loaded.")
print(f"LOCAL_ROOT: {LOCAL_ROOT}")
print(f"S3_ROOT: s3://{BUCKET}/{S3_ROOT}/")


Upload/download helpers loaded.
LOCAL_ROOT: /home/jovyan/scratch/eds-work-processing
S3_ROOT: s3://dcceew-eds-data/EDS-work-testing/eds-work-processing/


In [26]:
# Change this tile as needed
TILE = "p115r079"

In [27]:
# Check first
#list_s3(TILE)

# Dry run upload
#push_run(TILE, "run-auto-scale", dryrun=True)

# # Real upload
#push_run(TILE, "run-auto-scale")

# # Pull it back later
# pull_run(TILE, "run-auto-scale")

In [28]:

# 1. Check local files exist
check_local(TILE)

# 2. Check S3 target
#list_s3(TILE)

# 3. Dry run first
#push_tile(TILE, dryrun=True)

# 4. Real upload only after dry run looks right
push_tile(TILE, dryrun=False)

# 5. Confirm after upload
# list_s3(TILE)


Checking local path: /home/jovyan/scratch/eds-work-processing/p115r079
Files: 112
Size: 3.62 GB
First few files:
 - /home/jovyan/scratch/eds-work-processing/p115r079/run-auto-scale/legacy_outputs/sl8olre_p115r079_d2025112920260201_dll_e32749.tif
 - /home/jovyan/scratch/eds-work-processing/p115r079/run-auto-scale/legacy_outputs/sl8olre_p115r079_d2025112920260201_dll_log_e32749.json
 - /home/jovyan/scratch/eds-work-processing/p115r079/run-auto-scale/legacy_outputs/sl8olre_p115r079_d2025112920260201_dlj_e32749.tif
 - /home/jovyan/scratch/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_vi-ndvi_e32749_sr-auto-10000_base-nodataaware_ndviDiffStdErr_bins.csv
 - /home/jovyan/scratch/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_vi-ndvi_e32749_sr-auto-10000_base-nodataaware_ndviDiffStdErr.png
 - /home/jovyan/scratch/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_com

CompletedProcess(args=['aws', 's3', 'sync', '/home/jovyan/scratch/eds-work-processing/p115r079/', 's3://dcceew-eds-data/EDS-work-testing/eds-work-processing/p115r079/', '--only-show-errors', '--no-progress'], returncode=0, stdout='', stderr='')

In [29]:
# ============================================================
# VERIFY WHAT EXISTS IN S3 FOR A TILE
# ============================================================



list_s3(TILE)

aws s3 ls s3://dcceew-eds-data/EDS-work-testing/eds-work-processing/p115r079/ --recursive --human-readable --summarize
2026-05-12 00:19:50   16.0 MiB EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_combined_raw_e32749.tif
2026-05-12 00:19:50   27.6 KiB EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_vi-ndvi_e32749_sr-auto-10000_base-nodataaware_ndviDiffStdErr.png
2026-05-12 00:19:50    5.8 KiB EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_vi-ndvi_e32749_sr-auto-10000_base-nodataaware_ndviDiffStdErr_bins.csv
2026-05-12 00:19:50  272 Bytes EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_vi-ndvi_e32749_sr-auto-10000_base-nodataaware_ndviDiffStdErr_stats.csv
2026-05-12 00:19:50    2.2 KiB EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/dia

CompletedProcess(args=['aws', 's3', 'ls', 's3://dcceew-eds-data/EDS-work-testing/eds-work-processing/p115r079/', '--recursive', '--human-readable', '--summarize'], returncode=0, stdout='2026-05-12 00:19:50   16.0 MiB EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_combined_raw_e32749.tif\n2026-05-12 00:19:50   27.6 KiB EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_vi-ndvi_e32749_sr-auto-10000_base-nodataaware_ndviDiffStdErr.png\n2026-05-12 00:19:50    5.8 KiB EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_vi-ndvi_e32749_sr-auto-10000_base-nodataaware_ndviDiffStdErr_bins.csv\n2026-05-12 00:19:50  272 Bytes EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_vi-ndvi_e32749_sr-auto-10000_base-nodataaware_ndviDiffStdErr_stats.csv\n2026-05-12 00:19:50    2

In [30]:
# ============================================================
# VERIFY WHAT EXISTS IN S3 FOR ONE RUN
# ============================================================

RUN = "run-auto-scale"

list_s3(TILE, RUN)

aws s3 ls s3://dcceew-eds-data/EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/ --recursive --human-readable --summarize
2026-05-12 00:19:50   16.0 MiB EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_combined_raw_e32749.tif
2026-05-12 00:19:50   27.6 KiB EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_vi-ndvi_e32749_sr-auto-10000_base-nodataaware_ndviDiffStdErr.png
2026-05-12 00:19:50    5.8 KiB EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_vi-ndvi_e32749_sr-auto-10000_base-nodataaware_ndviDiffStdErr_bins.csv
2026-05-12 00:19:50  272 Bytes EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_vi-ndvi_e32749_sr-auto-10000_base-nodataaware_ndviDiffStdErr_stats.csv
2026-05-12 00:19:50    2.2 KiB EDS-work-testing/eds-work-processing/p115r079/run

CompletedProcess(args=['aws', 's3', 'ls', 's3://dcceew-eds-data/EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/', '--recursive', '--human-readable', '--summarize'], returncode=0, stdout='2026-05-12 00:19:50   16.0 MiB EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_combined_raw_e32749.tif\n2026-05-12 00:19:50   27.6 KiB EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_vi-ndvi_e32749_sr-auto-10000_base-nodataaware_ndviDiffStdErr.png\n2026-05-12 00:19:50    5.8 KiB EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_vi-ndvi_e32749_sr-auto-10000_base-nodataaware_ndviDiffStdErr_bins.csv\n2026-05-12 00:19:50  272 Bytes EDS-work-testing/eds-work-processing/p115r079/run-auto-scale/diagnostics/sl8olre_p115r079_d2025112920260201_vi-ndvi_e32749_sr-auto-10000_base-nodataaware_ndviDiffStdErr_stats.csv\n2026-05-1

## DELETE AFTER UPLOAD IF SURE

In [31]:
import shutil
from pathlib import Path


local_tile = Path("/home/jovyan/scratch/eds-work-processing") / TILE

print(f"Deleting: {local_tile}")

shutil.rmtree(local_tile)

print("Done")

Deleting: /home/jovyan/scratch/eds-work-processing/p115r079
Done


# PULL BACK FROM S3 TO HOME

In [35]:
pull_run(
    "p089r084",
    "run-auto-scale",
    dryrun=False
)

aws s3 sync s3://dcceew-eds-data/EDS-work-testing/eds-work-processing/p089r084/run-auto-scale/ /home/jovyan/scratch/eds-work-processing/p089r084/run-auto-scale/ --only-show-errors --no-progress


CompletedProcess(args=['aws', 's3', 'sync', 's3://dcceew-eds-data/EDS-work-testing/eds-work-processing/p089r084/run-auto-scale/', '/home/jovyan/scratch/eds-work-processing/p089r084/run-auto-scale/', '--only-show-errors', '--no-progress'], returncode=0, stdout='', stderr='')